In [1]:
import pandas as pd
import numpy as np
import os

RAW     = "raw_data"
CLEAN   = "cleaned_data"
os.makedirs(CLEAN, exist_ok=True)

SEP = "=" * 60

# ══════════════════════════════════════════════════════════════
# HELPER
# ══════════════════════════════════════════════════════════════
def section(title):
    print(f"\n{SEP}\n  {title}\n{SEP}")

def report(label, before, after):
    print(f"  [{label}]  before: {before:>7,}  →  after: {after:>7,}  (removed: {before - after:,})")

In [2]:
# ══════════════════════════════════════════════════════════════
# TABLE 1 — USERS
# ══════════════════════════════════════════════════════════════
section("TABLE 1 · users  →  users.csv")

users = pd.read_csv(f"{RAW}/streampulse_users.csv")
rows_start = len(users)
print(f"\n  Raw shape : {users.shape}")

# ── 1. Duplicates ──────────────────────────────────────────
dups = users.duplicated().sum()
users.drop_duplicates(inplace=True)
print(f"\n  Duplicates removed : {dups}")

# ── 2. Missing values ──────────────────────────────────────
print(f"\n  Nulls before:")
print(users.isnull().sum().to_string(header=False).replace("^", "    "))

# acquisition_channel: 191 nulls → fill with 'Unknown'
users['acquisition_channel'] = users['acquisition_channel'].fillna('Unknown')
print(f"\n  acquisition_channel nulls filled with 'Unknown'")

# ── 3. Date conversion ─────────────────────────────────────
users['signup_date'] = pd.to_datetime(users['signup_date'])
print(f"  signup_date converted to datetime")

# ── 4. Standardise text columns ────────────────────────────
users['city_tier']            = users['city_tier'].str.strip().str.title()
users['acquisition_channel']  = users['acquisition_channel'].str.strip().str.title()
print(f"  Text columns stripped & title-cased")

# ── 5. Validate user_id ────────────────────────────────────
before = len(users)
users = users[users['user_id'] > 0]
print(f"  Invalid user_ids removed : {before - len(users)}")

# ── 6. Final state ─────────────────────────────────────────
print(f"\n  Nulls after  :\n{users.isnull().sum().to_string()}")
print(f"\n  Final dtypes :\n{users.dtypes.to_string()}")
report("USERS ROWS", rows_start, len(users))

users.to_csv(f"{CLEAN}/users.csv", index=False)
print(f"\n  ✅ Saved → cleaned_data/users.csv")


  TABLE 1 · users  →  users.csv

  Raw shape : (10000, 4)

  Duplicates removed : 0

  Nulls before:
user_id                  0
signup_date              0
city_tier                0
acquisition_channel    191

  acquisition_channel nulls filled with 'Unknown'
  signup_date converted to datetime
  Text columns stripped & title-cased
  Invalid user_ids removed : 0

  Nulls after  :
user_id                0
signup_date            0
city_tier              0
acquisition_channel    0

  Final dtypes :
user_id                         int64
signup_date            datetime64[ns]
city_tier                      object
acquisition_channel            object
  [USERS ROWS]  before:  10,000  →  after:  10,000  (removed: 0)

  ✅ Saved → cleaned_data/users.csv


In [3]:
# ══════════════════════════════════════════════════════════════
# TABLE 2 — SUBSCRIPTIONS
# ══════════════════════════════════════════════════════════════
section("TABLE 2 · subscriptions  →  subscriptions.csv")

subs = pd.read_csv(f"{RAW}/streampulse_subscriptions.csv")
rows_start = len(subs)
print(f"\n  Raw shape : {subs.shape}")

# ── 1. Duplicates ──────────────────────────────────────────
dups = subs.duplicated().sum()
subs.drop_duplicates(inplace=True)
print(f"\n  Duplicates removed : {dups}")

# ── 2. Fix monthly_fee — stored as paise/cents (×100) ──────
#    Free=0, Basic=99→9.99, Premium=199→19.99
fee_map = {0: 0.00, 99: 9.99, 199: 19.99}
before_unique = subs['monthly_fee'].value_counts().to_dict()
print(f"\n  monthly_fee unique values (raw) : {before_unique}")

subs['monthly_fee'] = subs['monthly_fee'].map(fee_map)
# Any unmapped values → NaN; fix by deriving from plan_type
plan_fee_map = {'Free': 0.00, 'Basic': 9.99, 'Premium': 19.99}
mask = subs['monthly_fee'].isna()
subs.loc[mask, 'monthly_fee'] = subs.loc[mask, 'plan_type'].map(plan_fee_map)
print(f"  monthly_fee corrected to decimal (0 / 9.99 / 19.99)")
print(f"  Unmapped fees fixed via plan_type : {mask.sum()}")

# ── 3. Date conversion ─────────────────────────────────────
subs['churn_date'] = pd.to_datetime(subs['churn_date'], errors='coerce')
print(f"  churn_date converted to datetime (NaT for active users — expected)")

# ── 4. Validate: Active users should NOT have a churn_date ─
bad_active = (subs['status'] == 'Active') & (subs['churn_date'].notna())
subs.loc[bad_active, 'churn_date'] = pd.NaT
print(f"  Churn dates cleared for Active users : {bad_active.sum()}")

# ── 5. Validate: Churned users MUST have a churn_date ──────
bad_churned = (subs['status'] == 'Churned') & (subs['churn_date'].isna())
print(f"  Churned users missing churn_date : {bad_churned.sum()} (flagged)")

# ── 6. Standardise text ────────────────────────────────────
subs['plan_type'] = subs['plan_type'].str.strip().str.title()
subs['status']    = subs['status'].str.strip().str.title()

# ── 7. Final state ─────────────────────────────────────────
print(f"\n  Nulls after  :\n{subs.isnull().sum().to_string()}")
print(f"\n  Final dtypes :\n{subs.dtypes.to_string()}")
print(f"\n  Plan distribution:\n{subs['plan_type'].value_counts().to_string()}")
print(f"\n  Status distribution:\n{subs['status'].value_counts().to_string()}")
report("SUBSCRIPTIONS ROWS", rows_start, len(subs))

subs.to_csv(f"{CLEAN}/subscriptions.csv", index=False)
print(f"\n  ✅ Saved → cleaned_data/subscriptions.csv")


  TABLE 2 · subscriptions  →  subscriptions.csv

  Raw shape : (10000, 5)

  Duplicates removed : 0

  monthly_fee unique values (raw) : {0: 5027, 99: 2435, 199: 1838, 299: 700}
  monthly_fee corrected to decimal (0 / 9.99 / 19.99)
  Unmapped fees fixed via plan_type : 700
  churn_date converted to datetime (NaT for active users — expected)
  Churn dates cleared for Active users : 0
  Churned users missing churn_date : 0 (flagged)

  Nulls after  :
user_id           0
plan_type         0
monthly_fee     700
status            0
churn_date     8956

  Final dtypes :
user_id                 int64
plan_type              object
monthly_fee           float64
status                 object
churn_date     datetime64[ns]

  Plan distribution:
plan_type
Free       5027
Basic      2435
Premium    1838
Family      700

  Status distribution:
status
Active     8956
Churned    1044
  [SUBSCRIPTIONS ROWS]  before:  10,000  →  after:  10,000  (removed: 0)

  ✅ Saved → cleaned_data/subscriptions.csv


In [4]:
# ══════════════════════════════════════════════════════════════
# TABLE 3 — SESSIONS
# ══════════════════════════════════════════════════════════════
section("TABLE 3 · sessions  →  sessions.csv")

sess = pd.read_csv(f"{RAW}/streampulse_sessions.csv")
rows_start = len(sess)
print(f"\n  Raw shape : {sess.shape}")

# ── 1. Duplicates ──────────────────────────────────────────
dups = sess.duplicated().sum()
sess.drop_duplicates(inplace=True)
print(f"\n  Duplicates removed : {dups}")

# ── 2. Missing values ──────────────────────────────────────
print(f"\n  Nulls before:\n{sess.isnull().sum().to_string()}")

# duration_minutes nulls → fill with median (robust to outliers)
med_duration = sess['duration_minutes'].median()
sess['duration_minutes'] = sess['duration_minutes'].fillna(med_duration)
print(f"\n  duration_minutes: {1244} nulls filled with median = {med_duration:.2f} min")

# device_type nulls → fill with 'Unknown'
sess['device_type'] = sess['device_type'].fillna('Unknown')
print(f"  device_type: 2571 nulls filled with 'Unknown'")

# ── 3. Outlier handling — duration_minutes ─────────────────
#    Method: IQR-based capping (Winsorisation)
Q1  = sess['duration_minutes'].quantile(0.25)
Q3  = sess['duration_minutes'].quantile(0.75)
IQR = Q3 - Q1
lower = max(0, Q1 - 1.5 * IQR)   # sessions can't be negative
upper = Q3 + 1.5 * IQR

outliers_low  = (sess['duration_minutes'] < lower).sum()
outliers_high = (sess['duration_minutes'] > upper).sum()

sess['duration_minutes'] = sess['duration_minutes'].clip(lower=lower, upper=upper)
sess['duration_minutes'] = sess['duration_minutes'].round(2)

print(f"\n  Outlier treatment (IQR Winsorisation):")
print(f"    IQR range  : {lower:.2f} – {upper:.2f} min")
print(f"    Capped low : {outliers_low:,} rows  (zero / near-zero durations)")
print(f"    Capped high: {outliers_high:,} rows  (extreme long sessions)")

# ── 4. Date conversion ─────────────────────────────────────
sess['session_date'] = pd.to_datetime(sess['session_date'])
print(f"\n  session_date converted to datetime")

# ── 5. Validate: sessions must belong to known users ───────
valid_users = set(users['user_id'])
before = len(sess)
sess = sess[sess['user_id'].isin(valid_users)]
print(f"  Sessions with unknown user_id removed : {before - len(sess)}")

# ── 6. Standardise text ────────────────────────────────────
sess['device_type'] = sess['device_type'].str.strip().str.title()
print(f"  device_type stripped & title-cased")

# ── 7. Sort for readability ────────────────────────────────
sess.sort_values(['user_id', 'session_date'], inplace=True)
sess.reset_index(drop=True, inplace=True)

# ── 8. Final state ─────────────────────────────────────────
print(f"\n  Nulls after  :\n{sess.isnull().sum().to_string()}")
print(f"\n  Final dtypes :\n{sess.dtypes.to_string()}")
print(f"\n  duration_minutes stats:")
print(sess['duration_minutes'].describe().round(2).to_string())
print(f"\n  Device distribution:\n{sess['device_type'].value_counts().to_string()}")
report("SESSIONS ROWS", rows_start, len(sess))

sess.to_csv(f"{CLEAN}/sessions.csv", index=False)
print(f"\n  ✅ Saved → cleaned_data/sessions.csv")


  TABLE 3 · sessions  →  sessions.csv

  Raw shape : (126702, 5)

  Duplicates removed : 1254

  Nulls before:
session_id             0
user_id                0
session_date           0
duration_minutes    1223
device_type         2545

  duration_minutes: 1244 nulls filled with median = 24.00 min
  device_type: 2571 nulls filled with 'Unknown'

  Outlier treatment (IQR Winsorisation):
    IQR range  : 0.00 – 66.00 min
    Capped low : 0 rows  (zero / near-zero durations)
    Capped high: 5,900 rows  (extreme long sessions)

  session_date converted to datetime
  Sessions with unknown user_id removed : 0
  device_type stripped & title-cased

  Nulls after  :
session_id          0
user_id             0
session_date        0
duration_minutes    0
device_type         0

  Final dtypes :
session_id                   int64
user_id                      int64
session_date        datetime64[ns]
duration_minutes           float64
device_type                 object

  duration_minutes stats:
co

In [5]:
# ══════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ══════════════════════════════════════════════════════════════
section("CLEANING COMPLETE — SUMMARY")

print(f"""
  RAW FILE                              →   CLEAN FILE
  ─────────────────────────────────────────────────────────
  streampulse_users.csv        (10,000) →   users.csv         ({len(users):,})
  streampulse_subscriptions.csv(10,000) →   subscriptions.csv ({len(subs):,})
  streampulse_sessions.csv    (126,702) →   sessions.csv      ({len(sess):,})

  ISSUES FIXED
  ─────────────────────────────────────────────────────────
  ✔ Duplicates removed         (sessions: 1,254 rows)
  ✔ Nulls filled               (acquisition_channel, device_type → 'Unknown')
  ✔ Nulls filled               (duration_minutes → median)
  ✔ monthly_fee corrected      (integer paise → decimal USD)
  ✔ Outliers capped            (duration_minutes via IQR Winsorisation)
  ✔ Date columns converted     (signup_date, session_date, churn_date → datetime)
  ✔ churn_date logic validated (Active=NaT, Churned=date)
  ✔ Text standardised          (strip + title case across all tables)
  ✔ Orphan sessions removed    (user_ids not in users table)
  ✔ Files renamed              (short, clean, lowercase)

  OUTPUT FOLDER : cleaned_data/
  ─────────────────────────────────────────────────────────
  users.csv  |  subscriptions.csv  |  sessions.csv
""")


  CLEANING COMPLETE — SUMMARY

  RAW FILE                              →   CLEAN FILE
  ─────────────────────────────────────────────────────────
  streampulse_users.csv        (10,000) →   users.csv         (10,000)
  streampulse_subscriptions.csv(10,000) →   subscriptions.csv (10,000)
  streampulse_sessions.csv    (126,702) →   sessions.csv      (125,448)

  ISSUES FIXED
  ─────────────────────────────────────────────────────────
  ✔ Duplicates removed         (sessions: 1,254 rows)
  ✔ Nulls filled               (acquisition_channel, device_type → 'Unknown')
  ✔ Nulls filled               (duration_minutes → median)
  ✔ monthly_fee corrected      (integer paise → decimal USD)
  ✔ Outliers capped            (duration_minutes via IQR Winsorisation)
  ✔ Date columns converted     (signup_date, session_date, churn_date → datetime)
  ✔ churn_date logic validated (Active=NaT, Churned=date)
  ✔ Text standardised          (strip + title case across all tables)
  ✔ Orphan sessions removed  